# Polars：列を計算し、同じ答えになるか確かめる

対応する本文は [polars.tex](../chapters/polars.tex) である。コードセルを上から順にShift+Enterで実行する。各問の説明にある値や図と、セルの下に表示される結果を比べる。ファイルを作る問では、その直前に保存先を用意する。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/07_polars/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "07_polars"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/07_polars


## 入力の6件を表示する

次のセルは、pandasの課題と同じ6件をCSVへ保存して読み込む。`detector` は検出器名、`time_ns` は時刻 [ns]、`signal` は信号値、`valid` は有効かどうかを表す。

`df.schema` は列名と型の一覧である。続く表が6行4列で、4件目の信号値が欠損を表すnullになっていることを確認する。

In [2]:
import numpy as np
import polars as pl

CSV_PATH = WORK / "events.csv"
CSV_PATH.write_text('detector,time_ns,signal,valid\nA,1000,12.4,true\nB,1180,9.8,true\nA,1300,15.1,true\nB,1510,,false\nA,1760,11.0,true\nB,2050,13.2,true\n', encoding="utf-8")
df = pl.read_csv(CSV_PATH)
print(df.schema)
print(df)

Schema({'detector': String, 'time_ns': Int64, 'signal': Float64, 'valid': Boolean})
shape: (6, 4)
┌──────────┬─────────┬────────┬───────┐
│ detector ┆ time_ns ┆ signal ┆ valid │
│ ---      ┆ ---     ┆ ---    ┆ ---   │
│ str      ┆ i64     ┆ f64    ┆ bool  │
╞══════════╪═════════╪════════╪═══════╡
│ A        ┆ 1000    ┆ 12.4   ┆ true  │
│ B        ┆ 1180    ┆ 9.8    ┆ true  │
│ A        ┆ 1300    ┆ 15.1   ┆ true  │
│ B        ┆ 1510    ┆ null   ┆ false │
│ A        ┆ 1760    ┆ 11.0   ┆ true  │
│ B        ┆ 2050    ┆ 13.2   ┆ true  │
└──────────┴─────────┴────────┴───────┘


### まず一つの列へ1を足す

表に対する計算を一つ試す。次の `expr` は「信号値へ1を足す」という指定であり、答えの配列そのものではない。`df.select()` に渡して実行すると、先頭は13.4、空欄の行はnullとなる。`alias()` は結果の列名を付ける。

In [3]:
expr = pl.col("signal") + 1
print(expr)
print(df.select(expr.alias("signal_plus_one")))

[(col("signal")) + (dyn int: 1)]
shape: (6, 1)
┌─────────────────┐
│ signal_plus_one │
│ ---             │
│ f64             │
╞═════════════════╡
│ 13.4            │
│ 10.8            │
│ 16.1            │
│ null            │
│ 12.0            │
│ 14.2            │
└─────────────────┘


## 問1：列選択と行抽出を分ける

`pl.col("signal")` は `signal` 列に行う計算を指定するための式である。`select()` は列を選び、`filter()` は条件を満たす行を選ぶ。値が欠けた部分は `null` と表示される。比較結果が null の行も `filter` では残らないが、`is_not_null` を書くと有効条件を明示できる。Polars の `null` と浮動小数点の `NaN` は別であり、後者は必要に応じて `is_finite` で検査する。

この後のセルにある `assert 条件` は、結果を自動で確認する命令である。条件が違っていればエラーで止まり、条件を満たせば次へ進む。何も表示されなくても、エラーが出なければその確認は通っている。画面の数値や図も、説明に示した値と比べる。

In [4]:
selected = df.select("detector", "signal").filter(
    pl.col("signal").is_not_null() & (pl.col("signal") >= 12.0))
assert selected.height == 3
assert selected.filter(pl.col("detector") == "A").height == 2
print(selected)

shape: (3, 2)
┌──────────┬────────┐
│ detector ┆ signal │
│ ---      ┆ ---    │
│ str      ┆ f64    │
╞══════════╪════════╡
│ A        ┆ 12.4   │
│ A        ┆ 15.1   │
│ B        ┆ 13.2   │
└──────────┴────────┘


## 問2：群平均を各行に戻す

`over("detector")` は6行を保ち、各行が属する群の平均を返す。`group_by().agg()` は検出器ごとの2行へまとめる。件数は全行なら `pl.len()`、有効信号なら `pl.col("signal").count()` となる。

Aの平均は約12.8333、Bの有効値の平均は11.5となる。Aの最初の行の `signal_centered` は、信号12.4からAの平均を引いた約-0.4333となる。

In [5]:
centered = df.with_columns(
    (pl.col("signal") - pl.col("signal").mean().over("detector")).alias("signal_centered"))
summary = centered.group_by("detector").agg(
    pl.len().alias("rows"),
    pl.col("signal").count().alias("count"),
    pl.col("signal").mean().alias("mean"),
    pl.col("signal").max().alias("max"),
).sort("detector")
assert centered.height == 6 and summary.height == 2
assert summary["rows"].to_list() == [3, 3]
assert summary["count"].to_list() == [3, 2]
np.testing.assert_allclose(summary["mean"].to_numpy(), [38.5 / 3, 11.5])
np.testing.assert_allclose(centered.group_by("detector").agg(pl.col("signal_centered").mean())["signal_centered"].to_numpy(), 0, atol=1e-12)
print(centered)
print(summary)

shape: (6, 5)
┌──────────┬─────────┬────────┬───────┬─────────────────┐
│ detector ┆ time_ns ┆ signal ┆ valid ┆ signal_centered │
│ ---      ┆ ---     ┆ ---    ┆ ---   ┆ ---             │
│ str      ┆ i64     ┆ f64    ┆ bool  ┆ f64             │
╞══════════╪═════════╪════════╪═══════╪═════════════════╡
│ A        ┆ 1000    ┆ 12.4   ┆ true  ┆ -0.433333       │
│ B        ┆ 1180    ┆ 9.8    ┆ true  ┆ -1.7            │
│ A        ┆ 1300    ┆ 15.1   ┆ true  ┆ 2.266667        │
│ B        ┆ 1510    ┆ null   ┆ false ┆ null            │
│ A        ┆ 1760    ┆ 11.0   ┆ true  ┆ -1.833333       │
│ B        ┆ 2050    ┆ 13.2   ┆ true  ┆ 1.7             │
└──────────┴─────────┴────────┴───────┴─────────────────┘
shape: (2, 5)
┌──────────┬──────┬───────┬───────────┬──────┐
│ detector ┆ rows ┆ count ┆ mean      ┆ max  │
│ ---      ┆ ---  ┆ ---   ┆ ---       ┆ ---  │
│ str      ┆ u32  ┆ u32   ┆ f64       ┆ f64  │
╞══════════╪══════╪═══════╪═══════════╪══════╡
│ A        ┆ 3    ┆ 3     ┆ 12.833333 ┆ 1

## 問3：実行計画と結果を区別する

`scan_csv` の返り値は計画を保持する LazyFrame であり、`collect` が実行して DataFrame を返す。`explain` には、必要な列の選択やフィルターがどこへ移動したかが表示される。表示の書式自体は版で異なる。

このように処理の指定を蓄えておき、最後にまとめて実行する方法を遅延実行という。`explain()` の表示は処理内容を説明するものであり、集計値ではない。結果の表は `collect()` の後に表示され、AとBの平均を確認できる。

In [6]:
query = (pl.scan_csv(CSV_PATH).select("detector", "signal")
         .filter(pl.col("signal").is_not_null())
         .group_by("detector").agg(pl.col("signal").mean().alias("mean"))
         .sort("detector"))
print(query.explain())
lazy_result = query.collect()
assert isinstance(query, pl.LazyFrame) and isinstance(lazy_result, pl.DataFrame)
np.testing.assert_allclose(lazy_result["mean"].to_numpy(), summary["mean"].to_numpy())
print(lazy_result)

SORT BY [col("detector")]
  AGGREGATE[maintain_order: false]
    [col("signal").mean().alias("mean")] BY [col("detector")]
    FROM
    Csv SCAN [/Users/ikomae/src/python_test/tex/notebooks/_outputs/07_polars/events.csv]
    PROJECT 2/4 COLUMNS
    SELECTION: col("signal").is_not_null()
    ESTIMATED ROWS: 8
shape: (2, 2)
┌──────────┬───────────┐
│ detector ┆ mean      │
│ ---      ┆ ---       │
│ str      ┆ f64       │
╞══════════╪═══════════╡
│ A        ┆ 12.833333 │
│ B        ┆ 11.5      │
└──────────┴───────────┘


## 問4：結合し、件数を横持ちへ変える

キーが重複する右表では、1つの左行に複数行が対応して出力が増える。`validate="m:1"` はその誤りを検出する。pivot の前に群ごとの件数を求め、存在しない組合せを0で埋める。

ここでの横持ちは、検出器名を列見出しへ移すことをいう。pivotは、行に並んでいる項目を列へ移す表の組み替えである。

In [7]:
gain = pl.DataFrame({"detector": ["A", "B"], "gain": [1.02, 0.97]})
merged = df.join(gain, on="detector", how="left", validate="m:1").with_columns(
    (pl.col("signal") * pl.col("gain")).alias("signal_calibrated"))
wide = (df.group_by("detector", "valid").agg(pl.len().alias("count"))
        .pivot(index="detector", on="valid", values="count").fill_null(0).sort("detector"))
assert merged.height == 6
print(wide)
assert wide.filter(pl.col("detector") == "A")["true"][0] == 3
assert wide.filter(pl.col("detector") == "B")["false"][0] == 1

shape: (2, 3)
┌──────────┬──────┬───────┐
│ detector ┆ true ┆ false │
│ ---      ┆ ---  ┆ ---   │
│ str      ┆ u32  ┆ u32   │
╞══════════╪══════╪═══════╡
│ A        ┆ 3    ┆ 0     │
│ B        ┆ 2    ┆ 1     │
└──────────┴──────┴───────┘


## 問5：pandas と同じ条件で照合する

信号が有効な行を集計し、両方で標準偏差の自由度を1にする。群の行順は API によって保証が異なるため、照合前にキーで並べる。CSV を介した値と、元の欠損件数も確認する。

In [8]:
import pandas as pd

polars_summary = (df.filter(pl.col("valid") & pl.col("signal").is_not_null())
    .group_by("detector").agg(pl.col("signal").count().alias("count"),
        pl.col("signal").mean().alias("mean"), pl.col("signal").std(ddof=1).alias("std"))
    .sort("detector"))
polars_summary.write_csv(WORK / "polars_summary.csv")
loaded = pd.read_csv(WORK / "polars_summary.csv")
pdf = pd.read_csv(CSV_PATH)
expected = pdf.loc[pdf.valid & pdf.signal.notna()].groupby("detector", as_index=False).agg(
    count=("signal", "count"), mean=("signal", "mean"), std=("signal", "std"))
pd.testing.assert_frame_equal(loaded, expected, check_exact=False, rtol=1e-12, atol=1e-12)
assert df["signal"].null_count() == pdf.signal.isna().sum() == 1
print(loaded)

  detector  count       mean       std
0        A      3  12.833333  2.084067
1        B      2  11.500000  2.404163
